In [4]:
import os
from dotenv import load_dotenv

from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame

import plotly.express as px
import plotly.graph_objects as go

In [5]:
load_dotenv()

KEY = os.getenv("key")
SECRET = os.getenv("secret")

In [6]:
client = StockHistoricalDataClient(KEY, SECRET)

bars = client.get_stock_bars(
    StockBarsRequest(
        symbol_or_symbols=["AAPL"],
        timeframe=TimeFrame.Day,
        start="2023-01-01",
    )
)

df = bars.df

df.info()

<class 'pandas.DataFrame'>
MultiIndex: 922 entries, ('AAPL', Timestamp('2023-01-03 05:00:00+0000', tz='UTC')) to ('AAPL', Timestamp('2026-09-04 04:00:00+0000', tz='UTC'))
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   open         922 non-null    float64
 1   high         922 non-null    float64
 2   low          922 non-null    float64
 3   close        922 non-null    float64
 4   volume       922 non-null    float64
 5   trade_count  922 non-null    float64
 6   vwap         922 non-null    float64
dtypes: float64(7)
memory usage: 60.4+ KB


In [7]:
df.describe()

,open,high,low,close,volume,trade_count,vwap
count,922.000000,922.000000,922.000000,922.000000,9.220000e+02,9.220000e+02,922.000000
mean,218.743023,221.065246,216.720427,219.008134,5.632259e+07,6.613153e+05,218.969074
std,45.305649,45.866091,44.777223,45.353302,2.492314e+07,2.206701e+05,45.335862
min,126.010000,127.770000,124.170000,125.020000,1.791057e+07,2.459320e+05,125.660032
25%,182.522500,184.532500,181.057500,182.695000,4.235653e+07,5.263102e+05,182.773727
50%,213.960000,215.925000,211.865050,214.025000,5.039561e+07,6.088830e+05,213.981626
75%,252.148750,254.911250,248.787500,252.060000,6.314397e+07,7.345352e+05,251.912194
max,340.030000,344.569900,337.350100,340.080000,3.186799e+08,2.212850e+06,340.796598


In [10]:
print(df.isna().sum())

print(df.isna().sum().sum())

open           0
high           0
low            0
close          0
volume         0
trade_count    0
vwap           0
dtype: int64
0


In [13]:
print(df.index)

print(df.index.get_level_values("timestamp"))

MultiIndex([('AAPL', '2023-01-03 05:00:00+00:00'),
            ('AAPL', '2023-01-04 05:00:00+00:00'),
            ('AAPL', '2023-01-05 05:00:00+00:00'),
            ('AAPL', '2023-01-06 05:00:00+00:00'),
            ('AAPL', '2023-01-09 05:00:00+00:00'),
            ('AAPL', '2023-01-10 05:00:00+00:00'),
            ('AAPL', '2023-01-11 05:00:00+00:00'),
            ('AAPL', '2023-01-12 05:00:00+00:00'),
            ('AAPL', '2023-01-13 05:00:00+00:00'),
            ('AAPL', '2023-01-17 05:00:00+00:00'),
            ...
            ('AAPL', '2026-08-24 04:00:00+00:00'),
            ('AAPL', '2026-08-25 04:00:00+00:00'),
            ('AAPL', '2026-08-26 04:00:00+00:00'),
            ('AAPL', '2026-08-27 04:00:00+00:00'),
            ('AAPL', '2026-08-28 04:00:00+00:00'),
            ('AAPL', '2026-08-31 04:00:00+00:00'),
            ('AAPL', '2026-09-01 04:00:00+00:00'),
            ('AAPL', '2026-09-02 04:00:00+00:00'),
            ('AAPL', '2026-09-03 04:00:00+00:00'),
            ('A

ANALYSE

In [14]:
df.corr()

,open,high,low,close,volume,trade_count,vwap
open,1.000000,0.998778,0.999011,0.997730,-0.256769,0.204234,0.998635
high,0.998778,1.000000,0.998633,0.999049,-0.236964,0.225397,0.999533
low,0.999011,0.998633,1.000000,0.999002,-0.265428,0.189084,0.999495
close,0.997730,0.999049,0.999002,1.000000,-0.250012,0.205731,0.999740
volume,-0.256769,-0.236964,-0.265428,-0.250012,1.000000,0.650875,-0.250596
trade_count,0.204234,0.225397,0.189084,0.205731,0.650875,1.000000,0.206381
vwap,0.998635,0.999533,0.999495,0.999740,-0.250596,0.206381,1.000000


In [16]:
corr = df.corr()

px.imshow(
    corr,
    zmin=-1,
    zmax=1,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    title = "Correlation Matrix"
).show()

In [17]:
dfi = df.reset_index()

px.line(
    dfi,
    x="timestamp",
    y="close",
    title="AAPL Closing Prices",
).show()

In [18]:
returns = dfi["close"].pct_change().dropna()
px.histogram(
    returns,
    nbins=50,
    title="AAPL Daily Returns Distribution",).show()

In [19]:
px.box(
    returns,
    title="AAPL Daily Returns Box Plot"
).show()

In [25]:
go.Figure(go.Candlestick(
    x=dfi["timestamp"],
    open=dfi["open"],
    high=dfi["high"],
    low=dfi["low"],
    close=dfi["close"],
)).show()